# Hawking radiation and black-hole evaporation

*astro · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/astro/hawking_radiation)

A black hole glows like a black body at the Hawking temperature $T_H = \hbar c^3/(8\pi G M k_B)$. One broadcast evaluates Planck's law for many masses and wavelengths at once, and RK4 integrates the mass loss $dM/dt = -\alpha/M^2$: the hole shrinks, heats up and ends in a burst after $t \propto M^3$.

**Vector form**

$$T_H = \frac{\hbar c^3}{8\pi G M k_B},\qquad P = \sigma T_H^4\,4\pi r_s^2 = \frac{\hbar c^6}{15360\,\pi G^2 M^2},\qquad \frac{dM}{dt} = -\frac{P}{c^2},\qquad t_\mathrm{life} = \frac{5120\,\pi G^2 M_0^3}{\hbar c^4}$$

**Index (tensor) form**

$$T_m = \frac{\hbar c^3}{8\pi G k_B}\,M_m^{-1},\qquad B_{ml} = \frac{2hc^2}{\lambda_l^5}\,\Big(e^{hc/(\lambda_l k_B T_m)} - 1\Big)^{-1},\qquad \dot M_m = -\alpha\,M_m^{-2}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $M_m$ | `masses` | (B,) | a batch of black-hole masses (kg): Sun, Earth, …, the animated hole |
| $T_m$ | `hawking_temperature(M)` | (B,) | Hawking temperature, elementwise |
| $\lambda_l$ | `wavelengths(...)` | (L,) | log-spaced wavelength grid |
| $B_{ml}$ | `planck(lam, T)` | (F, L) | Planck spectrum by broadcasting T[:, None] against lam[None, :] |
| $\lambda_\mathrm{max}$ | `wien_peak(T)` | (B,) | Wien's law $b/T_H$ |
| $P_m$ | `power(M)` | (B,) | Stefan–Boltzmann power of the horizon |
| $\dot M_m$ | `mass_loss_rate(M)` | (B,) | $-P/c^2$, integrated with RK4 |
| $t_\mathrm{life}$ | `lifetime(M0)` | (B,) | analytic lifetime, compared with the integration |

SI units throughout. A solar-mass black hole is at $6\times10^{-8}$ K, far colder than the 2.7 K microwave background, so real astrophysical black holes absorb more than they emit. Only tiny (primordial) holes evaporate: one of about $2\times10^{11}$ kg formed in the Big Bang would be finishing now. The model counts photons only and treats the horizon as a perfect black body of area $4\pi r_s^2$; real holes also emit neutrinos, gravitons and, when hot enough, heavier particles, which shortens the lifetime, and greybody factors change the spectrum's shape. The time step is chosen so that each RK4 step burns a fixed fraction of the remaining mass; the animation advances in equal steps of $\log M$, so the last frames cover a tiny time span. Shown with the horizon radius drawn as $M^{0.35}$ so it stays visible.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/astro/hawking_radiation/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    M0 = 10 ** params["log10_mass"]
    names, masses = tensors.example_masses(M0, dev)

    T = equation.hawking_temperature(masses)  # (B,) one temperature per mass, all at once
    t_life = equation.lifetime(masses)
    print(f"{'black hole':>26}  {'mass (kg)':>10}  {'T_H (K)':>10}  {'λ_max (m)':>10}  {'lifetime (s)':>12}")
    for name, M, T_b, lam_b, t_b in zip(names, masses, T, equation.wien_peak(T), t_life):
        print(f"{name:>26}  {M:10.2e}  {T_b:10.2e}  {lam_b:10.2e}  {t_b:12.2e}")

    times, history = solve.evaporate(masses, fraction=params["burn_fraction"])  # (S, B) each
    numeric = times[-1] / t_life
    print(f"integrated lifetime / formula 5120πG²M³/(ħc⁴): {numeric.min():.4f} … {numeric.max():.4f}")

    h = history[:, -1]  # the animated hole
    T_h = equation.hawking_temperature(h)
    lam = tensors.wavelengths(0.03 * float(equation.wien_peak(T_h[-1])), 30 * float(equation.wien_peak(T_h[0])),
                              300, dev)  # fmt: skip
    return plot.animate(times[:, -1], h, T_h, equation.power(h), lam, float(t_life[-1]),
                        out_dir / "hawking_radiation.gif")  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/astro/hawking_radiation/equation.py
r"""Physics → tensor translation.

Quantum fields near a horizon make a black hole glow like a black body (Hawking 1974) at the temperature

    T_H = \frac{\hbar c^3}{8\pi G M k_B}           (smaller hole ⇒ hotter)

Its spectrum is Planck's law, evaluated for a batch of masses m and wavelengths l at once by broadcasting:

    B_{ml} = \frac{2 h c^2}{λ_l^5}\,\frac{1}{e^{hc/(λ_l k_B T_m)} - 1}

and peaks at Wien's λ_max = b / T_H. Treating the horizon (area 4π r_s², r_s = 2GM/c²) as a black body
radiating photons, the Stefan–Boltzmann law gives the power and the mass loss:

    P = σ T_H^4 \, 4π r_s^2 = \frac{\hbar c^6}{15360\,π G^2 M^2},     \frac{dM}{dt} = -\frac{P}{c^2} = -\frac{α}{M^2}

Integrating, M^3 falls linearly in time and the hole vanishes after

    t_{life} = \frac{M_0^3}{3α} = \frac{5120\,π G^2 M_0^3}{\hbar c^4}.

The last moments are a burst: P ∝ 1/M² grows without bound as M → 0.
"""

import math

import torch

# SI constants
HBAR = 1.054571817e-34  # J s
H = 2 * math.pi * HBAR
C = 2.99792458e8  # m/s
G = 6.67430e-11  # m³/(kg s²)
KB = 1.380649e-23  # J/K
SIGMA = 5.670374419e-8  # W/(m² K⁴)
WIEN_B = 2.897771955e-3  # m K
M_SUN = 1.98847e30  # kg


def hawking_temperature(M: torch.Tensor) -> torch.Tensor:
    r"""T_H = ħc³ / (8π G M k_B) for every mass, (...) → (...) in kelvin."""
    return HBAR * C**3 / (8 * math.pi * G * M * KB)


def schwarzschild_radius(M: torch.Tensor) -> torch.Tensor:
    return 2 * G * M / C**2


def planck(lam: torch.Tensor, T: torch.Tensor) -> torch.Tensor:
    r"""Spectral radiance B_{ml} for masses m (temperatures T_m) and wavelengths λ_l.   (L,), (M,) → (M, L)"""
    x = H * C / (lam[None, :] * KB * T[:, None])  # (M, L) broadcast: one row per black hole
    return 2 * H * C**2 / lam[None, :] ** 5 / torch.expm1(x)


def wien_peak(T: torch.Tensor) -> torch.Tensor:
    """λ_max = b / T."""
    return WIEN_B / T


def power(M: torch.Tensor) -> torch.Tensor:
    r"""P = σ T_H⁴ · 4π r_s² (photons only), in watts."""
    return SIGMA * hawking_temperature(M) ** 4 * 4 * math.pi * schwarzschild_radius(M) ** 2


def mass_loss_rate(M: torch.Tensor) -> torch.Tensor:
    r"""dM/dt = −P/c² = −α/M²."""
    return -power(M) / C**2


def lifetime(M0: torch.Tensor) -> torch.Tensor:
    r"""t_life = 5120 π G² M_0³ / (ħ c⁴), in seconds."""
    return 5120 * math.pi * G**2 * M0**3 / (HBAR * C**4)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/astro/hawking_radiation/tensors.py
"""Build the concrete tensors: a batch of black-hole masses and a logarithmic wavelength grid (SI units)."""

import torch

from .equation import M_SUN

EXAMPLES = {  # name → mass in kg, for the printed table
    "Sun": M_SUN,
    "Earth": 5.97e24,
    "Moon": 7.35e22,
    "Mount Everest": 8.1e14,
    "dying today (primordial)": 1.7e11,
}


def example_masses(M0: float, device: torch.device) -> tuple[list[str], torch.Tensor]:
    """Names and masses (B,) in kg: a few famous ones plus the hole we animate."""
    names = [*EXAMPLES, "animated hole"]
    masses = torch.tensor([*EXAMPLES.values(), M0], dtype=torch.float64, device=device)
    return names, masses


def wavelengths(lam_min: float, lam_max: float, n: int, device: torch.device) -> torch.Tensor:
    """λ grid (L,) spaced evenly in log λ, in metres."""
    return torch.logspace(torch.log10(torch.tensor(lam_min)), torch.log10(torch.tensor(lam_max)), n,
                          dtype=torch.float64, device=device)  # fmt: skip


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/astro/hawking_radiation/solve.py
"""Integrate the evaporation dM/dt = −α/M² with RK4, letting each step remove a fixed fraction of the mass."""

import torch

from problems._common.progress import steps

from .equation import mass_loss_rate


def rk4_step(M: torch.Tensor, dt: torch.Tensor) -> torch.Tensor:
    """One RK4 step of dM/dt = f(M) (f does not depend on t), elementwise for a batch of holes."""
    k1 = mass_loss_rate(M)
    k2 = mass_loss_rate(M + 0.5 * dt * k1)
    k3 = mass_loss_rate(M + 0.5 * dt * k2)
    k4 = mass_loss_rate(M + dt * k3)
    return M + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)


def evaporate(M0: torch.Tensor, fraction: float = 0.01, M_end: float = 1e-3) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T, B), masses (T, B)) from M0 down to M_end·M0.

    The time step shrinks with the hole: dt = fraction · M / |dM/dt|, so every step burns about the same
    fraction of the remaining mass. A fixed dt would either crawl at the start or overshoot in the final burst.
    """
    n_steps = int(torch.log(torch.tensor(1 / M_end)) / fraction) + 1
    t, M = torch.zeros_like(M0), M0.clone()
    times, masses = [t], [M]
    for _ in steps(n_steps, "RK4 steps"):
        dt = fraction * M / mass_loss_rate(M).abs()
        M = rk4_step(M, dt)
        t = t + dt
        times.append(t)
        masses.append(M)
    return torch.stack(times), torch.stack(masses)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/astro/hawking_radiation/plot.py
"""Animate a small black hole evaporating: shrinking horizon, hotter and bluer spectrum, final burst."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .equation import planck, wien_peak


def animate(times: torch.Tensor, masses: torch.Tensor, T: torch.Tensor, P: torch.Tensor, lam: torch.Tensor,
            t_life: float, path: Path) -> Path:  # fmt: skip
    """times, masses, T, P: (S,) histories of the animated hole; lam: wavelength grid (L,)."""
    frames = frame_indices(len(times), 85)
    steps = torch.tensor(frames)
    spectra = planck(lam, T[steps]).cpu().numpy()  # (F, L): one Planck curve per frame, by broadcasting
    t = times.cpu().numpy() / t_life
    m = (masses / masses[0]).cpu().numpy()
    T_np, P_np, lam_np = T.cpu().numpy(), P.cpu().numpy(), lam.cpu().numpy()
    peaks = wien_peak(T).cpu().numpy()
    burst = [frames[-1]] * 12  # hold the end with a flash

    fig = plt.figure(figsize=(8, 4.5))
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.25])
    ax = fig.add_subplot(gs[:, 0])
    ax_s = fig.add_subplot(gs[0, 1])
    ax_m = fig.add_subplot(gs[1, 1])

    # --- the hole itself: radius ∝ M, glow colour from the temperature ---
    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-2.1, 1.6)
    ax.set_aspect("equal")
    ax.axis("off")
    cmap = plt.cm.inferno
    logT = np.log10(T_np)
    shade = lambda k: cmap(0.25 + 0.7 * (logT[k] - logT[0]) / max(1e-9, logT[-1] - logT[0]))  # noqa: E731
    glow = plt.Circle((0, 0), 1.25, color=shade(0), alpha=0.5)
    hole = plt.Circle((0, 0), 1.0, color="black")
    ax.add_patch(glow)
    ax.add_patch(hole)
    rays = [ax.plot([], [], color="#f59e0b", lw=2)[0] for _ in range(16)]
    info = ax.text(0, -2.05, "", ha="center", va="bottom", fontsize=8, family="monospace")
    ax.set_title("black hole (drawn with size ∝ M^0.35)", fontsize=9)

    # --- spectra ---
    ax_s.set_xscale("log")
    ax_s.set_yscale("log")
    ax_s.set_xlim(lam_np[0], lam_np[-1])
    top = spectra.max()
    ax_s.set_ylim(top * 1e-14, top * 5)
    ax_s.set_xlabel("wavelength λ (m)", fontsize=8)
    ax_s.set_ylabel("B_λ (W sr⁻¹ m⁻³)", fontsize=8)
    ax_s.tick_params(labelsize=7)
    ghosts = [ax_s.plot(lam_np, s, color="#d1d5db", lw=0.6)[0] for s in spectra]
    for g in ghosts:
        g.set_visible(False)
    now = ax_s.plot([], [], color=ACCENT[1], lw=2)[0]
    peak = ax_s.axvline(peaks[0], color=ACCENT[0], ls="--", lw=1)
    ax_s.set_title("Planck spectrum at T_H, dashed: Wien peak b/T_H", fontsize=8)

    # --- mass history ---
    s = np.linspace(0, 1, 400)
    ax_m.plot(s, (1 - s) ** (1 / 3), color="#9ca3af", lw=1, label="(1 − t/t_life)^{1/3}")
    run = ax_m.plot([], [], "o", ms=2.5, color=ACCENT[0], label="RK4")[0]
    ax_m.set_xlim(0, 1.02)
    ax_m.set_ylim(0, 1.05)
    ax_m.set_xlabel("t / t_life", fontsize=8)
    ax_m.set_ylabel("M / M₀", fontsize=8)
    ax_m.tick_params(labelsize=7)
    ax_m.legend(fontsize=7, loc="lower left")

    shown = {"n": 0}

    def update(step: int):
        k = frames.index(step)
        final = step == frames[-1] and shown["n"] >= len(frames)
        shown["n"] += 1
        r = m[step] ** 0.35  # gentle compression so the hole stays visible to the end
        hole.set_radius(r)
        glow.set_radius(r * 1.25 + (0.6 if final else 0.0))
        glow.set_color(shade(step))
        glow.set_alpha(0.9 if final else 0.5)
        angles = np.linspace(0, 2 * np.pi, len(rays), endpoint=False)
        reach = 1.5 if final else r * 1.25 + 0.25 * (1 - m[step])
        for ray, a in zip(rays, angles):
            ray.set_data([r * 1.3 * np.cos(a), reach * np.cos(a)], [r * 1.3 * np.sin(a), reach * np.sin(a)])
            ray.set_visible(final or reach > r * 1.35)
        for j, g in enumerate(ghosts):
            g.set_visible(j < k and j % 6 == 0)  # every 6th earlier spectrum
        now.set_data(lam_np, spectra[k])
        peak.set_xdata([peaks[step], peaks[step]])
        run.set_data(t[: step + 1], m[: step + 1])
        left = (1 - t[step]) * t_life
        info.set_text(
            f"M = {masses[step]:.2e} kg   T_H = {T_np[step]:.2e} K\nP = {P_np[step]:.2e} W   time left {left:.2e} s"
            + ("\nfinal burst!" if final else "")
        )

    fig.tight_layout()
    return save_gif(fig, update, frames + burst, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'log10_mass': 5.4,
    'burn_fraction': 0.01
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.astro.hawking_radiation")]:
    del sys.modules[name]
main = importlib.import_module("problems.astro.hawking_radiation.main")
problem_dir = Path("problems/astro/hawking_radiation")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
